# **`Inatel - C318 (Tópicos Especiais II) - 2026/1`**

# <font color='green'>**Atividade 07: HPO e AutoML**</font>

## <font color='#2D9CDB'>**LEIA ATENTAMENTE AS INSTRUÇÕES A SEGUIR**</font>
- Importe este notebook no [Google Colab](https://colab.research.google.com/) para resolver os exercícios;
- Consulte a apostila disponibilizada pelo professor para se familiarizar com os conceitos;
- Utilize os recursos disponíveis na Internet (documentações e artigos científicos) para complementar seus estudos;
- <font color='red'>**Uso consciente de Inteligência Artificial (LLMs):**</font>
  - O uso de assistentes (como Gemini, ChatGPT, Claude) é permitido, mas exige responsabilidade técnica:
    - Em vez de pedir a solução completa, peça para a IA explicar conceitos, sugerir abordagens ou ajudar a depurar erros de código;
    - Você é o responsável por cada linha de código entregue. Não insira no notebook implementações que você não compreende integralmente ou não saberia explicar;
    - Modelos de linguagem podem "alucinar" funções ou sugerir métodos obsoletos de bibliotecas em Python. Sempre teste e verifique a documentação oficial;
    - Quando utilizar a IA para gerar ou refatorar blocos lógicos complexos, indique isso através de comentários no próprio código;
- <font color='red'>**NÃO**</font> remova as células de Código já presentes neste notebook;
- <font color='red'>**NÃO**</font> modifique as células de Markdown (em <font color='green'>verde</font> ou <font color='#2D9CDB'>azul</font>) presentes neste notebook;
- Após cada questão, há uma célula para você implementar e responder a questão;
- É permitido adicionar mais células (de código ou markdown) antes da próxima pergunta;
- Caso precise utilizar bibliotecas que não estão instaladas nativamente no Colab, inclua uma célula de código com o comando de instalação (ex: `!pip install nome_da_biblioteca`);
- <font color='red'>**Renomeie o termo `_Enunciado` para `_seu_numero_de_matricula` no nome do arquivo (exemplo: `C318_2026_1_Atividade_07_12345.ipynb`)**</font>;
- <font color='magenta'>**Faça download do notebook com a resolução no Google Colab, mantendo a saída de todas as células, e anexe-o à tarefa do Teams.**</font>

# <font color='green'><u><b>Preparação</b></u></font>

In [17]:
!pip install numpy pandas matplotlib seaborn scikit-learn optuna

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os

np.random.seed(42)

# <font color='green'><u><b>Parte 1 - Otimização de Hiperparâmetros (HPO)</b></u></font>

### <font color='#2D9CDB'>Q2) Escolha um problema de classificação disponível no módulo <a href="https://scikit-learn.org/stable/api/sklearn.datasets.html" target="blank">sklearn.datasets</a> ou no repositório <a href="https://archive.ics.uci.edu/datasets/?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true" target="blank">UCI Machine Learning Repository</a>.</font>
<ul>
    <font color='#2D9CDB'><li>Utilizando um modelo de regressão diferente daquele empregado nos exemplos da apostila, realize a otimização de hiperparâmetros com Optuna.</li></font>
    <font color='#2D9CDB'><li>Defina um espaço de busca contendo pelo menos três hiperparâmetros, utilize validação cruzada para avaliação e execute pelo menos 30 trials.</li></font>
    <font color='#2D9CDB'><li>Apresente os melhores hiperparâmetros encontrados, a métrica de classificação utilizada e o desempenho obtido pelo modelo otimizado.</li></font>
    <font color='#2D9CDB'><li>Em seguida, compare os resultados com aqueles obtidos utilizando a configuração padrão do algoritmo e discuta os ganhos observados.</li></font>
</ul>
<font color='2D9CDB'>Não é permitido utilizar o dataset Digits nem o mesmo modelo apresentado nos exemplos da apostila.</font>

In [19]:
from sklearn.datasets import load_wine

# Carrega o dataset
wine = load_wine()

X = wine.data

print("Formato:", X.shape)

Formato: (178, 13)


### <font color='#2D9CDB'>Q2) Escolha um problema de regressão disponível no módulo <a href="https://scikit-learn.org/stable/api/sklearn.datasets.html" target="blank">sklearn.datasets</a> ou no repositório <a href="https://archive.ics.uci.edu/datasets/?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true" target="blank">UCI Machine Learning Repository</a>.</font>
<ul>
    <font color='#2D9CDB'><li>Utilizando um modelo de classificação diferente daquele empregado nos exemplos da apostila, realize a otimização de hiperparâmetros com Optuna.</li></font>
    <font color='#2D9CDB'><li>Defina um espaço de busca contendo pelo menos três hiperparâmetros, utilize validação cruzada para avaliação e execute pelo menos 30 trials.</li></font>
    <font color='#2D9CDB'><li>Apresente os melhores hiperparâmetros encontrados, a métrica de classificação utilizada e o desempenho obtido pelo modelo otimizado.</li></font>
    <font color='#2D9CDB'><li>Em seguida, compare os resultados com aqueles obtidos utilizando a configuração padrão do algoritmo e discuta os ganhos observados.</li></font>
</ul>
<font color='2D9CDB'>Não é permitido utilizar o mesmo dataset nem o mesmo modelo apresentado nos exemplos da apostila.</font>

In [13]:
from sklearn.datasets import load_breast_cancer

# Carrega o dataset
dataset = load_breast_cancer()

X = dataset.data
y = dataset.target

print(X.shape)
print("Classes:", dataset.target_names)

(569, 30)
Classes: ['malignant' 'benign']


In [14]:
import optuna

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    n_estimators = trial.suggest_int(
        "n_estimators",
        50,
        300
    )

    max_depth = trial.suggest_int(
        "max_depth",
        2,
        30
    )

    min_samples_split = trial.suggest_int(
        "min_samples_split",
        2,
        20
    )

    min_samples_leaf = trial.suggest_int(
        "min_samples_leaf",
        1,
        10
    )

    max_features = trial.suggest_categorical(
        "max_features",
        ["sqrt", "log2", None]
    )

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    score = cross_val_score(
        clf,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    ).mean()

    return score

study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=30
)

print("Melhores hiperparâmetros:")
print(study.best_params)

print("\nMelhor acurácia:")
print(study.best_value)

[I 2026-06-26 22:17:22,253] A new study created in memory with name: no-name-15f667da-90b2-4c0f-9127-abe1eb594f53
[I 2026-06-26 22:17:23,796] Trial 0 finished with value: 0.9473218444340941 and parameters: {'n_estimators': 50, 'max_depth': 17, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9473218444340941.
[I 2026-06-26 22:17:34,031] Trial 1 finished with value: 0.9455519329296692 and parameters: {'n_estimators': 261, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 9, 'max_features': None}. Best is trial 0 with value: 0.9473218444340941.
[I 2026-06-26 22:17:37,541] Trial 2 finished with value: 0.9490607048594939 and parameters: {'n_estimators': 54, 'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 2 with value: 0.9490607048594939.
[I 2026-06-26 22:17:41,001] Trial 3 finished with value: 0.9525694767893185 and parameters: {'n_estimators': 170, 'max_depth': 8, 'min_samp

Melhores hiperparâmetros:
{'n_estimators': 117, 'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None}

Melhor acurácia:
0.9578326346840551


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

modelo_padrao = RandomForestClassifier(
    random_state=42
)

modelo_otimizado = RandomForestClassifier(
    **study.best_params,
    random_state=42
)

score_padrao = cross_val_score(
    modelo_padrao,
    X,
    y,
    cv=cv,
    scoring="accuracy"
).mean()

score_otimizado = cross_val_score(
    modelo_otimizado,
    X,
    y,
    cv=cv,
    scoring="accuracy"
).mean()

print(f"Acurácia (modelo padrão): {score_padrao:.4f}")
print(f"Acurácia (modelo otimizado): {score_otimizado:.4f}")

Modelo	//               Acurácia

Random Forest (padrão)	95,61%

Random Forest (otimizado)	95,78%

Observa-se que a otimização proporcionou um pequeno ganho de desempenho, aumentando a acurácia de 95,61% para 95,78%, um incremento de aproximadamente 0,17 ponto percentual. Embora a melhoria seja modesta, ela demonstra que o processo de otimização foi capaz de encontrar uma configuração ligeiramente mais adequada para esse conjunto de dados.

### <font color='#2D9CDB'>Q3) Escolha um problema de agrupamento disponível no módulo <a href="https://scikit-learn.org/stable/api/sklearn.datasets.html" target="blank">sklearn.datasets</a> ou no repositório <a href="https://archive.ics.uci.edu/datasets/?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true" target="blank">UCI Machine Learning Repository</a>.</font>
<ul>
    <font color='#2D9CDB'><li>Utilizando um modelo de agrupamento diferente daquele empregado nos exemplos da apostila, realize a otimização de hiperparâmetros com Optuna.</li></font>
    <font color='#2D9CDB'><li>Defina um espaço de busca contendo pelo menos três hiperparâmetros, utilize validação cruzada para avaliação e execute pelo menos 30 trials.</li></font>
    <font color='#2D9CDB'><li>Apresente os melhores hiperparâmetros encontrados, a métrica de classificação utilizada e o desempenho obtido pelo modelo otimizado.</li></font>
    <font color='#2D9CDB'><li>Em seguida, compare os resultados com aqueles obtidos utilizando a configuração padrão do algoritmo e discuta os ganhos observados.</li></font>
</ul>
<font color='2D9CDB'>Não é permitido utilizar o mesmo dataset nem o mesmo modelo apresentado nos exemplos da apostila.</font><br>
<font color='2D9CDB'>Também é possível escolher um conjuntos de dados de classificação ou regressão e descartar a variável-alvo.</font>

# <font color='green'><u><b>Parte 2 - Aprendizado de Máquina Automatizado (AutoML)</b></u></font>

### <font color='#2D9CDB'>Q4) Leia o artigo científico abaixo. Em seguida, responda às questões a seguir, fundamentando suas respostas com evidências apresentadas no artigo:
<ol>
<font color='#2D9CDB'><li>O que é AutoML e quais etapas do pipeline de Machine Learning podem ser automatizadas;</li></font>
<font color='#2D9CDB'><li>Qual a diferença entre Hyperparameter Optimization (HPO) e AutoML;</li></font>
<font color='#2D9CDB'><li>Qual foi o principal objetivo do estudo;</li></font>
<font color='#2D9CDB'><li>Quais ferramentas apresentaram melhor equilíbrio entre desempenho preditivo e custo computacional;</li></font>
<font color='#2D9CDB'><li>Quais são as principais vantagens e limitações das abordagens AutoML discutidas pelos autores.</li></font>
</ol>

<font color='#2D9CDB'><i>Aragão, M.V.C., Afonso, A.G., Ferraz, R.C. et al. A practical evaluation of AutoML tools for binary, multiclass, and multilabel classification. Nature, Scientific Reports, 17682 (2025). DOI: <a href="https://doi.org/10.1038/s41598-025-02149-x" target="blank">10.1038/s41598-025-02149-x</a></i></font>

1) AutoML (Automated Machine Learning) é um conjunto de técnicas e ferramentas que automatizam o processo de construção de modelos de aprendizado de máquina, reduzindo a necessidade de intervenção humana e de conhecimento especializado. O objetivo é facilitar o desenvolvimento de modelos com bom desempenho, automatizando tarefas que normalmente exigiriam ajustes manuais.

2) A Hyperparameter Optimization (HPO) corresponde apenas à etapa de busca dos melhores hiperparâmetros para um algoritmo específico. Nesse caso, o modelo já foi escolhido previamente e a otimização procura encontrar a configuração que maximize seu desempenho.
Já o AutoML possui um escopo muito mais amplo. Além da otimização de hiperparâmetros, ele automatiza outras etapas do pipeline, como pré-processamento dos dados, seleção de atributos, escolha do algoritmo, treinamento, validação e, em alguns casos, construção de ensembles. Dessa forma, a HPO é apenas um dos componentes que podem fazer parte de um sistema AutoML.

3) O principal objetivo do estudo foi realizar uma avaliação prática e abrangente de ferramentas AutoML para problemas de classificação binária, multiclasse e multirrótulo (multilabel).

4) As ferramentas que demonstraram o melhor equilíbrio entre desempenho preditivo e custo computacional foram principalmente:

AutoSklearn

PyCaret

AutoGluon

5) Vantagens das ferramentas AutoML:

automatização de grande parte do pipeline de Machine Learning;

redução da necessidade de conhecimento especializado;

elevada capacidade de encontrar modelos com bom desempenho;

maior produtividade no desenvolvimento de modelos;

facilidade para comparar diferentes algoritmos e configurações automaticamente;

boa capacidade de adaptação a diferentes tipos de problemas de classificação.

Limitações importantes:

elevado custo computacional de algumas ferramentas, especialmente aquelas que utilizam buscas exaustivas;

diferenças significativas entre as ferramentas quanto ao suporte para classificação multirrótulo;

dependência da qualidade dos dados de entrada;

algumas ferramentas fazem fortes suposições sobre o pré-processamento dos dados, exigindo entradas numéricas ou previamente tratadas;

sensibilidade ao hardware disponível, principalmente em ferramentas que utilizam aceleração por GPU;

variabilidade entre execuções devido ao uso de métodos estocásticos;

não existe uma ferramenta AutoML que seja superior em todos os cenários, sendo a escolha dependente do tipo de problema, do conjunto de dados e dos recursos computacionais disponíveis.

### <font color='#2D9CDB'>Q5) Escolha uma ferramenta AutoML avaliada no artigo de Aragão et al. (2025), como AutoGluon, AutoSklearn, PyCaret, FLAML, TPOT, H2O AutoML ou outra ferramenta analisada pelos autores.</font>
<ol>
<font color='#2D9CDB'><li>Utilize essa ferramenta para resolver novamente o problema de regressão ou classificação desenvolvido na Questão 1 ou na Questão 2.</li></font>
<font color='#2D9CDB'><li>Execute o processo completo de AutoML e apresente o modelo vencedor, a métrica obtida, o tempo total de execução e os principais hiperparâmetros da solução final.</li></font>
<font color='#2D9CDB'><li>Compare os resultados obtidos com aqueles alcançados utilizando Optuna na questão correspondente.</li></font>
<font color='#2D9CDB'><li>Discuta as vantagens e desvantagens observadas, a facilidade de utilização da ferramenta, o custo computacional envolvido e se os resultados obtidos justificam o uso de AutoML para o problema analisado.</li></font>
</ol>
</font>

In [11]:
!pip install flaml -q

In [12]:
import time
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from flaml import AutoML

# Dataset
dataset = load_breast_cancer()

X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

automl = AutoML()

settings = {
    "time_budget": 60,          # segundos
    "metric": "accuracy",
    "task": "classification",
    "seed": 42
}

inicio = time.time()

automl.fit(
    X_train=X_train,
    y_train=y_train,
    **settings
)

fim = time.time()

pred = automl.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Modelo vencedor:", automl.model.estimator)

print("\nAcurácia:", acc)

print("\nTempo total:", round(fim-inicio,2), "segundos")

print("\nMelhores hiperparâmetros:")

print(automl.best_config)

[flaml.automl.logger: 06-26 22:12:34] {2375} INFO - task = classification
[flaml.automl.logger: 06-26 22:12:34] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 06-26 22:12:34] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 06-26 22:12:34] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 06-26 22:12:34] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 06-26 22:12:34] {3046} INFO - Estimated sufficient time budget=2062s. Estimated necessary time budget=48s.
[flaml.automl.logger: 06-26 22:12:34] {3097} INFO -  at 0.2s,	estimator lgbm's best error=9.0380e-02,	best estimator lgbm's best error=9.0380e-02
[flaml.automl.logger: 06-26 22:12:34] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 06-26 22:12:34] {3097} INFO -  at 0.8s,	estimator lgbm's best error=9.0380e-02,	best estimator lgbm's best error=9.0380e-02
[flaml.auto

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 06-26 22:12:34] {3097} INFO -  at 1.0s,	estimator sgd's best error=3.7184e-01,	best estimator lgbm's best error=9.0380e-02
[flaml.automl.logger: 06-26 22:12:34] {2911} INFO - iteration 3, current learner sgd
[flaml.automl.logger: 06-26 22:12:35] {3097} INFO -  at 1.0s,	estimator sgd's best error=3.7184e-01,	best estimator lgbm's best error=9.0380e-02
[flaml.automl.logger: 06-26 22:12:35] {2911} INFO - iteration 4, current learner lgbm
[flaml.automl.logger: 06-26 22:12:35] {3097} INFO -  at 1.2s,	estimator lgbm's best error=8.7911e-02,	best estimator lgbm's best error=8.7911e-02
[flaml.automl.logger: 06-26 22:12:35] {2911} INFO - iteration 5, current learner sgd
[flaml.automl.logger: 06-26 22:12:35] {3097} INFO -  at 1.9s,	estimator sgd's best error=1.9095e-01,	best estimator lgbm's best error=8.7911e-02
[flaml.automl.logger: 06-26 22:12:35] {2911} INFO - iteration 6, current learner xgboost
[flaml.automl.logger: 06-26 22:12:36] {3097} INFO -  at 2.3s,	estimator xg

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 06-26 22:13:34] {3097} INFO -  at 60.1s,	estimator lrl1's best error=9.0348e-02,	best estimator lgbm's best error=2.0095e-02
[flaml.automl.logger: 06-26 22:13:34] {3359} INFO - retrain lgbm for 0.0s
[flaml.automl.logger: 06-26 22:13:34] {3362} INFO - retrained model: LGBMClassifier(colsample_bytree=np.float64(0.45266031641078613),
               learning_rate=np.float64(0.7513665424930773), max_bin=7,
               min_child_samples=28, n_estimators=21, n_jobs=-1, num_leaves=22,
               reg_alpha=np.float64(0.006036089679019034),
               reg_lambda=np.float64(0.008042958127754575), verbose=-1)
[flaml.automl.logger: 06-26 22:13:34] {2636} INFO - fit succeeded
[flaml.automl.logger: 06-26 22:13:34] {2637} INFO - Time taken to find the best model: 40.31810402870178
Modelo vencedor: LGBMClassifier(colsample_bytree=np.float64(0.45266031641078613),
               learning_rate=np.float64(0.7513665424930773), max_bin=7,
               min_child_samples=28, 